# Example-06: FFRFT/ZOOM frequency estimation

In [1]:
# Import

import numpy
import torch
import nufft
import yaml

import sys
sys.path.append('..')

from harmonica.util import LENGTH
from harmonica.statistics import weighted_mean, weighted_variance
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())
print(torch.get_num_threads())

True
16


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# FFRFT frequency estimation is based on local DTFT spectrum interpolation (near expected global maximum)
# By default refined spectum is computed inside two FFT bins around maximum bin
# In this case expected frequency error is proportional to 1/n^2

# Set parameters (64 signals with length 1024)

size, length = 64, 1024

# Set window

w = Window.from_cosine(length, order=1.0, dtype=dtype, device=device)
print(w)

# Set TbT data (64 signals with two components and different amplitudes)

t = torch.linspace(1.0, length, length, dtype=dtype, device=device)
data = torch.stack([i*torch.sin(2.0*numpy.pi*1*0.12*t) + 0.01*i*torch.sin(2.0*numpy.pi*2*0.12*t) for i in range(1, size + 1)])
d = Data.from_data(w, data)
print(d)

# Initialize Frequency instance

f1 = Frequency(d)
f2 = Frequency(d, pad=8192, fraction=1.0)
print(f1)
print(f2)

# Apply window (note, window is applied to work)

d.window_remove_mean()
d.window_apply()

# Fraction between 1.0-2.0 should be save to use

# Estimate frequency, see also task_ffrft

f1('ffrft')
f2('ffrft')

# Compare results FFT & FFRFT estimations

print(torch.abs(f1.fft_frequency.mean() - 0.12))
print(torch.abs(f2.fft_frequency.mean() - 0.12))
print(torch.abs(f1.ffrft_frequency.mean() - 0.12))
print(torch.abs(f2.ffrft_frequency.mean() - 0.12))

# Clean

del w
del t, data
del d
del f1, f2
if device != torch.device('cpu'):
    torch.cuda.synchronize()
    torch.cuda.empty_cache()

Window(1024, 'cosine_window', 1.0)
Data(64, Window(1024, 'cosine_window', 1.0))
Frequency(Data(64, Window(1024, 'cosine_window', 1.0)), f_range=(0.0, 0.5))
Frequency(Data(64, Window(1024, 'cosine_window', 1.0)), f_range=(0.0, 0.5))
tensor(1.171875000000e-04, dtype=torch.float64)
tensor(4.882812499996e-06, dtype=torch.float64)
tensor(8.392333984419e-07, dtype=torch.float64)
tensor(1.144409179643e-07, dtype=torch.float64)
